# Project 5: Ensemble Models and Wine Quality Predictions
**Author:** Alissa Beaderstadt<br>
**Date:** November 16, 2025<br>

## Introduction
In this project, we explore ensemble models, a powerful approach in machine learning that combines multiple models to improve performance.

## Imports
Import the necessary Python libraries for this notebook.  

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    BaggingClassifier,
    VotingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

## Section 1. Load and Inspect the Data


In [2]:
# Load Titanic dataset
df = pd.read_csv("winequality-red.csv", sep=";")

# Display structure and first few rows
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


`Note:`<br>
-  The dataset includes 11 features
-  The target variable is:
   -  quality (integer score from 0 to 10, rated by wine tasters)
- We will simplify this target into three categories:
  - low (3–4), medium (5–6), high (7–8) to make classification feasible.
  - we will also make this numeric (we want both for clarity)
- The dataset contains 1599 samples and 12 columns (11 features + target).

## Section 2. Prepare the Data
Includes cleaning, feature engineering, encoding, splitting, helper functions

### Convert quality values into labels
We group the scores into three categories (low, medium, high) because wine quality is subjective and easier to interpret as groups rather than individual numeric values.

In [4]:
def quality_to_label(q):
    if q <= 4:
        return "low"
    elif q <= 6:
        return "medium"
    else:
        return "high"


### Add the new column quality_label
Create a new column named `quality_label` using the function above.

In [5]:
df["quality_label"] = df["quality"].apply(quality_to_label)
df.head()


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,quality_label
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,medium
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,medium
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,medium
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,medium
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,medium


### Convert labels into numeric form for modeling
Machine learning algorithms need numeric input, so we convert low, medium, high to 0, 1, 2.

In [6]:
def quality_to_number(q):
    if q <= 4:
        return 0
    elif q <= 6:
        return 1
    else:
        return 2

df["quality_numeric"] = df["quality"].apply(quality_to_number)


## Section 3. Feature Selection and Justification
**Target:** `quality_label` (the new column we just created)<br>
**Features:** all columns except 'quality' and 'quality_label' and 'quality_numberic' - drop these from the input array


- Explain / introduce your choices.

In [7]:
X = df.drop(columns=["quality", "quality_label", "quality_numeric"])  # Features
y = df["quality_numeric"]  # Target

## Section 4. Split the Data into Train and Test

In [8]:
# Train/test split (stratify to preserve class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Section 5.  Evaluate Model Performance (Choose 2)
Now that the data is prepared and split, we will test two different ensemble model approaches to compare their performance.

In [10]:
# Helper function to train and evaluate models
def evaluate_model(name, model, X_train, y_train, X_test, y_test, results):
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    train_f1 = f1_score(y_train, y_train_pred, average="weighted")
    test_f1 = f1_score(y_test, y_test_pred, average="weighted")

    print(f"\n{name} Results")
    print("Confusion Matrix (Test):")
    print(confusion_matrix(y_test, y_test_pred))
    print(f"Train Accuracy: {train_acc:.4f}, Test Accuracy: {test_acc:.4f}")
    print(f"Train F1 Score: {train_f1:.4f}, Test F1 Score: {test_f1:.4f}")

    results.append(
        {
            "Model": name,
            "Train Accuracy": train_acc,
            "Test Accuracy": test_acc,
            "Train F1": train_f1,
            "Test F1": test_f1,
        }
    )

### Models selected for comparison
- For this project I chose to compare:
  - 1	Random Forest (100)	A strong baseline model using 100 decision trees.
  - 6	Voting (DT + SVM + NN)	Combines diverse models by averaging their predictions.

In [11]:
results = []

# 1. Random Forest (100)
evaluate_model(
    "Random Forest (100)",
    RandomForestClassifier(n_estimators=100, random_state=42),
    X_train,
    y_train,
    X_test,
    y_test,
    results,
)

# 6. Voting Classifier (DT + SVM + NN)
voting = VotingClassifier(
    estimators=[
        ("DT", DecisionTreeClassifier()),
        ("SVM", SVC(probability=True)),
        ("NN", MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000)),
    ],
    voting="soft",
)

evaluate_model(
    "Voting (DT + SVM + NN)",
    voting,
    X_train,
    y_train,
    X_test,
    y_test,
    results,
)



Random Forest (100) Results
Confusion Matrix (Test):
[[  0  13   0]
 [  0 256   8]
 [  0  15  28]]
Train Accuracy: 1.0000, Test Accuracy: 0.8875
Train F1 Score: 1.0000, Test F1 Score: 0.8661

Voting (DT + SVM + NN) Results
Confusion Matrix (Test):
[[  0  13   0]
 [  0 250  14]
 [  0  19  24]]
Train Accuracy: 0.9234, Test Accuracy: 0.8562
Train F1 Score: 0.9054, Test F1 Score: 0.8351


### Model Comparison
- Random Forest (100) did better overall than the Voting Classifier.
  - Test Accuracy: 0.8875 vs 0.8562
  - Test F1: 0.8661 vs 0.8351

- Random Forest got more of the harder-to-classify wines right (third class: 28 vs 24).
  - Training accuracy was perfect for Random Forest (1.0) vs 0.9234 for Voting, it was a little overfit, but still strong on test data.

**Overall:** Random Forest is the stronger model, it is more accurate and better balanced, even if slightly overfit.

## Section 6. Compare Results 

In [12]:
# Create a table of results 
results_df = pd.DataFrame(results)

print("\nSummary of All Models:")
display(results_df)


Summary of All Models:


,Model,Train Accuracy,Test Accuracy,Train F1,Test F1
0,Random Forest (100),1.000000,0.88750,1.000000,0.866056
1,Voting (DT + SVM + NN),0.923378,0.85625,0.905361,0.835124


### Understanding the gaps:

- Gap = Train score - Test score
- Big gap = possible overfitting (model learned training too well)
- Small gap = generalizes nicely to new data
- Checking gaps + test accuracy = fastest way to spot the best models

In [13]:
results_df['Accuracy Gap'] = results_df['Train Accuracy'] - results_df['Test Accuracy']
results_df['F1 Gap'] = results_df['Train F1'] - results_df['Test F1']

# Sort by Test Accuracy to quickly see the best model
results_df = results_df.sort_values(by='Test Accuracy', ascending=False)
display(results_df)


,Model,Train Accuracy,Test Accuracy,Train F1,Test F1,Accuracy Gap,F1 Gap
0,Random Forest (100),1.000000,0.88750,1.000000,0.866056,0.112500,0.133944
1,Voting (DT + SVM + NN),0.923378,0.85625,0.905361,0.835124,0.067128,0.070237


## Section 7. Conclusions and Insights

Using both your results and the results from others, which options are performing well and why do you think so. 

This is your value as an analyst - narrate your story, link to other notebooks, provide a comprehensive view of what you feel is the best model for predicting quality in red wine. Base all your reasoning on data. Feel free to tune parameters if you like.  Discuss the types of models and why you think some seem to be more helpful. List the next steps you'd like to try if you were in a competition to build the best predictor. 

Don't just copy code and don't just copy AI insights - use them to learn, but we all get them for free. Use all your tools to provide your own unique value and insights. Professional communication skills are critical. Evaluate your work in the context of others - how well can you craft a unique data story and present a compelling project to your clients / readers / self. 